# Controlling Output

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [47]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, model=model, system=None, temperature=0.0, **arg):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    params.update(arg)

    message = client.messages.create(**params)
    return message.content[0].text

In [13]:
messages = []
add_user_message (messages, "Generate a very short event bridge rule as json")
chat(messages)

'```json\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [16]:
messages = []

add_user_message (messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences = ["```"])
text

'\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [ ]:
import json

# Clean up and parse the JSON
json.loads(text.strip())

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

## Recap

#### The Solution: Assistant Message Prefilling + Stop Sequences

To generate structured data output (e.g. Json, python code, Bulleted lists, CSV data, etc.), you can combine **assistant message prefilling** with **stop sequences** to get exactly the content you want.

#### This technique works by:

1. The user message tells Claude what to generate
2. The prefilled assistant message makes Claude think **it already started a markdown code block**
3. Claude continues by writing just the JSON content
4. When Claude tries to close the code block with ```, **the stop sequence immediately ends generation**

#### Tips:
The key is identifying what Claude naturally wants to wrap your content in, then using that as your prefill and stop sequence. For code, it's usually markdown code blocks. For lists, it might be different formatting markers.



In [18]:
# Exercise!

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message (messages, prompt)

text = chat(messages)
text.strip()

'# Three Sample AWS CLI Commands\n\n1. **List all S3 buckets:**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 instances:**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID:**\n```bash\naws sts get-caller-identity\n```'

In [ ]:
from IPython.display import Markdown

Markdown(text)

# Three Sample AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```

In [54]:
# implementing a solution for the Exercise (no change of the prompt)
# note: model behaves differently (haiku vs. sonnet; sonnet-4-5 vs. sonnet-4-6)
# claude-sonnet-4-6 doesn't support assistant message prefill –
# it must end with a user message

# Exercise!

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message (messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(
    messages,
    model = "claude-sonnet-4-5",
    stop_sequences = ["```"]
    )
text.strip()

'aws s3 ls\n\naws ec2 describe-instances --region us-east-1\n\naws iam list-users'

In [55]:
Markdown(text)


aws s3 ls

aws ec2 describe-instances --region us-east-1

aws iam list-users


In [57]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message (messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(
    messages,
    model = "claude-haiku-4-5",
    stop_sequences = ["```"]
    )
Markdown(text)


# 1. List all S3 buckets
aws s3 ls

# 2. Describe EC2 instances
aws ec2 describe-instances

# 3. Get current AWS account ID
aws sts get-caller-identity


In [60]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message (messages, prompt)
add_assistant_message(
    messages, 
    "Here are all three commands in a single block without any comments: \n```bash"
)

text = chat(
    messages,
    model = "claude-haiku-4-5",
    stop_sequences = ["```"]
    )


Markdown(text)


aws s3 ls
aws ec2 describe-instances
aws dynamodb list-tables
